In [7]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import (
    precision_score,
    recall_score,
    roc_auc_score,
    accuracy_score,
    f1_score
)
from imblearn.over_sampling import RandomOverSampler

import warnings
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [ ]:
# Model and validation settings
logreg = LogisticRegression(max_iter=5000)

n_splits = 5
seed = 10

# Stratified split keeps the class ratio similar across folds
kf = StratifiedShuffleSplit(
    n_splits=n_splits,
    test_size=1 / n_splits,
    random_state=seed
)



In [ ]:
# Prepare fold-level data
def prepare_fold_data(X_train, X_val):
    X_train = X_train.copy()
    X_val = X_val.copy()

    # Convert boolean columns to integers for model input
    bool_cols_train = X_train.select_dtypes(include=["bool"]).columns
    bool_cols_val = X_val.select_dtypes(include=["bool"]).columns

    if len(bool_cols_train) > 0:
        X_train[bool_cols_train] = X_train[bool_cols_train].astype(int)
    if len(bool_cols_val) > 0:
        X_val[bool_cols_val] = X_val[bool_cols_val].astype(int)

    # Apply dummy encoding and align columns between train and validation sets
    X_train = pd.get_dummies(X_train, drop_first=False)
    X_val = pd.get_dummies(X_val, drop_first=False)

    X_train, X_val = X_train.align(X_val, join="left", axis=1, fill_value=0)

    return X_train, X_val



In [ ]:
# Cross-validation function
def cross_validate_model(prefix, upsample=True):
    X = pd.read_csv(f"data/{prefix}_X_train.csv", keep_default_na=False)
    y = pd.read_csv(f"data/{prefix}_y_train.csv", keep_default_na=False).values.ravel()

    cv_scores = {
        "fold": [],
        "test_precision": [],
        "test_recall": [],
        "test_roc_auc": [],
        "test_accuracy": [],
        "test_f1": []
    }

    splits = list(kf.split(X, y))

    for fold_idx, (train_idx, val_idx) in enumerate(splits):
        X_train = X.iloc[train_idx].copy()
        y_train = y[train_idx].copy()

        X_val = X.iloc[val_idx].copy()
        y_val = y[val_idx].copy()

        # Apply oversampling only to the training fold
        if upsample:
            upsampler = RandomOverSampler(random_state=seed)
            X_train, y_train = upsampler.fit_resample(X_train, y_train)

            if not isinstance(X_train, pd.DataFrame):
                X_train = pd.DataFrame(X_train, columns=X.columns)

        X_train_prepared, X_val_prepared = prepare_fold_data(X_train, X_val)

        clf = logreg.fit(X_train_prepared, y_train)

        y_pred = clf.predict(X_val_prepared)
        y_prob = clf.predict_proba(X_val_prepared)[:, 1]

        cv_scores["fold"].append(fold_idx + 1)
        cv_scores["test_precision"].append(precision_score(y_val, y_pred))
        cv_scores["test_recall"].append(recall_score(y_val, y_pred))
        cv_scores["test_roc_auc"].append(roc_auc_score(y_val, y_prob))
        cv_scores["test_accuracy"].append(accuracy_score(y_val, y_pred))
        cv_scores["test_f1"].append(f1_score(y_val, y_pred))

    for key in ["test_precision", "test_recall", "test_roc_auc", "test_accuracy", "test_f1"]:
        cv_scores[key] = np.array(cv_scores[key])

    summary_df = pd.DataFrame({
        "fold": cv_scores["fold"],
        "precision": cv_scores["test_precision"],
        "recall": cv_scores["test_recall"],
        "roc_auc": cv_scores["test_roc_auc"],
        "accuracy": cv_scores["test_accuracy"],
        "f1": cv_scores["test_f1"]
    })

    # Mean performance across CV folds
    mean_row = pd.DataFrame([{
        "fold": "mean",
        "precision": summary_df["precision"].mean(),
        "recall": summary_df["recall"].mean(),
        "roc_auc": summary_df["roc_auc"].mean(),
        "accuracy": summary_df["accuracy"].mean(),
        "f1": summary_df["f1"].mean()
    }])
    # Calculate SE
    n = n_splits

    # Cross-validated standard error
    se_row = pd.DataFrame([{
        "fold": "se",
        "precision": summary_df["precision"].std(ddof=1) / np.sqrt(n),
        "recall": summary_df["recall"].std(ddof=1) / np.sqrt(n),
        "roc_auc": summary_df["roc_auc"].std(ddof=1) / np.sqrt(n),
        "accuracy": summary_df["accuracy"].std(ddof=1) / np.sqrt(n),
        "f1": summary_df["f1"].std(ddof=1) / np.sqrt(n)
    }])


    summary_df = pd.concat([summary_df, mean_row, se_row], ignore_index=True)

    print(f"Results for {prefix} (upsample={upsample})")

    for metric in ["precision", "recall", "roc_auc", "accuracy", "f1"]:
        mean_value = summary_df.loc[summary_df["fold"] == "mean", metric].values[0]
        se_value = summary_df.loc[summary_df["fold"] == "se", metric].values[0]
        print(f"Mean {metric}: {mean_value:.3f} ± {se_value:.3f}")

    return cv_scores, summary_df

In [11]:
cv_scores_mask_before, summary_mask_before = cross_validate_model("mask_before", upsample=True)
summary_mask_before

Results for mask_before (upsample=True)
Mean precision: 0.481 ± 0.003
Mean recall: 0.757 ± 0.007
Mean roc_auc: 0.818 ± 0.003
Mean accuracy: 0.725 ± 0.003
Mean f1: 0.588 ± 0.003


,fold,precision,recall,roc_auc,accuracy,f1
0,1,0.486842,0.764706,0.826296,0.730008,0.594929
1,2,0.470705,0.753577,0.810936,0.716406,0.579462
2,3,0.488325,0.764706,0.824454,0.731245,0.596035
3,4,0.483193,0.731320,0.813609,0.727535,0.581910
4,5,0.475910,0.769475,0.812256,0.720528,0.588092
5,mean,0.480995,0.756757,0.817510,0.725144,0.588086
6,se,0.003350,0.006875,0.003252,0.002866,0.003336


In [ ]:
# Run Logistic Regression for all tasks
cv_scores_mask_before, summary_mask_before = cross_validate_model("mask_before", upsample=True)
cv_scores_mask_after, summary_mask_after = cross_validate_model("mask_after", upsample=True)
cv_scores_general_before, summary_general_before = cross_validate_model("general_before", upsample=True)
cv_scores_general_after, summary_general_after = cross_validate_model("general_after", upsample=True)

Results for mask_before (upsample=True)
Mean precision: 0.481 ± 0.003
Mean recall: 0.757 ± 0.007
Mean roc_auc: 0.818 ± 0.003
Mean accuracy: 0.725 ± 0.003
Mean f1: 0.588 ± 0.003
Results for mask_after (upsample=True)
Mean precision: 0.883 ± 0.001
Mean recall: 0.782 ± 0.002
Mean roc_auc: 0.838 ± 0.002
Mean accuracy: 0.773 ± 0.001
Mean f1: 0.830 ± 0.001
Results for general_before (upsample=True)
Mean precision: 0.692 ± 0.002
Mean recall: 0.700 ± 0.006
Mean roc_auc: 0.737 ± 0.003
Mean accuracy: 0.676 ± 0.003
Mean f1: 0.696 ± 0.004
Results for general_after (upsample=True)
Mean precision: 0.859 ± 0.002
Mean recall: 0.726 ± 0.004
Mean roc_auc: 0.779 ± 0.001
Mean accuracy: 0.715 ± 0.003
Mean f1: 0.787 ± 0.003


In [ ]:
# Build comparison table
def get_mean_se(summary_df, metric):
    mean_value = summary_df.loc[summary_df["fold"] == "mean", metric].values[0]
    se_value = summary_df.loc[summary_df["fold"] == "se", metric].values[0]
    return mean_value, se_value


summaries = {
    "mask_before": summary_mask_before,
    "mask_after": summary_mask_after,
    "general_before": summary_general_before,
    "general_after": summary_general_after
}

rows = []

for model_name, summary in summaries.items():
    row = {"model": model_name}
    
    for metric in ["precision", "recall", "roc_auc", "accuracy", "f1"]:
        mean_value, se_value = get_mean_se(summary, metric)
        row[metric] = mean_value
        row[f"{metric}_se"] = se_value
        row[f"{metric}_mean_se"] = f"{mean_value:.3f} ± {se_value:.3f}"
    
    rows.append(row)

comparison_df = pd.DataFrame(rows)

comparison_df

,model,precision,precision_se,precision_mean_se,recall,recall_se,recall_mean_se,roc_auc,roc_auc_se,roc_auc_mean_se,accuracy,accuracy_se,accuracy_mean_se,f1,f1_se,f1_mean_se
0,mask_before,0.480995,0.003350,0.481 ± 0.003,0.756757,0.006875,0.757 ± 0.007,0.817510,0.003252,0.818 ± 0.003,0.725144,0.002866,0.725 ± 0.003,0.588086,0.003336,0.588 ± 0.003
1,mask_after,0.882814,0.001163,0.883 ± 0.001,0.782499,0.002360,0.782 ± 0.002,0.838444,0.001598,0.838 ± 0.002,0.772835,0.001102,0.773 ± 0.001,0.829622,0.001064,0.830 ± 0.001
2,general_before,0.692239,0.002500,0.692 ± 0.002,0.700000,0.005758,0.700 ± 0.006,0.736907,0.002628,0.737 ± 0.003,0.676010,0.003180,0.676 ± 0.003,0.696065,0.003687,0.696 ± 0.004
3,general_after,0.858983,0.001877,0.859 ± 0.002,0.725777,0.003834,0.726 ± 0.004,0.778811,0.001346,0.779 ± 0.001,0.714943,0.003009,0.715 ± 0.003,0.786762,0.002599,0.787 ± 0.003


In [16]:
pd.DataFrame(comparison_df).to_csv("results/lr_mask_before.csv", index=False)